## Notebook to learn to play with tif images

In [ ]:
import sys
import importlib as imp
import numpy as np
import matplotlib.pyplot as plt
import rasterio

import experiment_settings
import build_model
import train_model
import build_data

import tensorflow as tf

# tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on

In [ ]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"tensorflow version = {tf.__version__}")  

print(tf.config.list_physical_devices('GPU'))

In [ ]:
EXP_NAME = "exp0"
settings = experiment_settings.get_settings(EXP_NAME)
# display(settings)

In [ ]:
imp.reload(build_data)

(tagyear_train, 
 taglat_train, 
 taglon_train,
 tagyear_val, 
 taglat_val, 
 taglon_val, 
 ) = build_data.make_sample_list(settings)

tfds_train = build_data.build_tf_dataset(settings, tagyear_train, taglat_train, taglon_train, settings["batch_size"])
tfds_val = build_data.build_tf_dataset(settings, tagyear_val, taglat_val, taglon_val, settings["batch_size"])

batch_shape = np.shape(next(tfds_val.as_numpy_iterator())[0])
print(f"{batch_shape = }")

In [ ]:
imp.reload(build_model)
imp.reload(train_model)

model = build_model.build_model(settings, input_shape=batch_shape[1:])

model, fit_summary, history, settings = train_model.train_model(settings, model, tfds_train, tfds_val)

fit_summary